In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision.transforms import v2

In [ ]:


# yuup some dummy nn model 
class MLP(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=128, num_classes=10):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = x.flatten(start_dim=1)
        return self.fc2(F.relu(self.fc1(x)))

In [ ]:
model = MLP().to("cuda")
named_params = list(model.named_parameters())
names = [name for name, _ in named_params]
shapes = [param.shape for _, param in named_params]
sizes = [param.numel() for _, param in named_params]

theta0 = torch.cat([
    param.detach().reshape(-1) for _, param in named_params
])
D = theta0.numel()
def pack(named_params, names, shapes, sizes, params):

    dct = {}
    offset = 0
    for idx, name in enumerate(names):
        dct[name] = params[offset:offset+sizes[idx]].reshape(shapes[idx])
        offset += sizes[idx]
    return dct
        

    

In [ ]:
# shapes[0], sizes[0]

In [ ]:
# print(dict(model.named_parameters()))

In [ ]:
BASELINE = True
d = 128
assert d<=D

# we do P (D, d) to make a possibility of update the directions latern on 
# so this is to what dimensions we want to cast our z-degree space.
P = torch.randn(D, d, device="cuda")
P = F.normalize(P, p=2, dim=0)

training_data = torchvision.datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)
# print(len(training_data[0]))
train_loader = torch.utils.data.DataLoader(training_data, batch_size=256, shuffle=True)
test_data = torchvision.datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=128, shuffle=True)
z = torch.zeros(d).to("cuda").detach().requires_grad_(True)

print(z.requires_grad)
print(z.is_leaf)
print(f"degrees of freedom: {d}")
if BASELINE:
    model = MLP().to("cuda")
    optim = torch.optim.AdamW(model.parameters())
else:
    optim = torch.optim.AdamW([z])
epochs = 10
for epoch in range(epochs):
    for x, y in train_loader:
        x, y = x.to("cuda"), y.to("cuda")


        optim.zero_grad()
        pred = None
        # so basically we are making the P the avaiable dimension of freedom z can move to. And we do that to fit the match the model dims
        if not BASELINE:
            theta = theta0 + P @ z
            theta_dict = pack(named_params, names, shapes, sizes, theta)
        
            pred = torch.func.functional_call(model, theta_dict, x)
        else: # baseline
            pred = model.forward(x) #torch.func.functional_call(model, model.parameters(), x)
        y = torch.nn.functional.one_hot(y, num_classes=10).float()
        loss = torch.nn.functional.mse_loss(pred, y)

        loss.backward()
        optim.step()
        # print(f"loss: {loss:.4f}")
    with torch.no_grad():
        total = torch.tensor(0.0)
        total_correct = torch.tensor(0.0)
        
        for x, y in test_loader:
            x, y = x.to("cuda"), y.to("cuda")

            pred = None
            if not BASELINE:
                # so basically we are making the P the avaiable dimension of freedom z can move to. And we do that to fit the match the model dims
                theta = theta0 + P @ z
                theta_dict = pack(named_params, names, shapes, sizes, theta)
                
                pred = torch.func.functional_call(model, theta_dict, x)
            else: #baseline
                pred = model.forward(x)
            y = y.float()
            # loss = torch.nn.functional.mse_loss(pred, y)

            pred = torch.argmax(pred, dim=1)
            # print(pred, y)
            num_correct = (pred == y).float().sum()
            # print(res.item(),  y.numel())
            total += y.numel()
            total_correct += num_correct.item()
        print(f"acc: {(total_correct / total):.5f}")


# baseline
# acc: 0.83080
# acc: 0.85580
# acc: 0.85860
# acc: 0.86500
# acc: 0.86980
# acc: 0.87200
# acc: 0.87390
# acc: 0.87400
# acc: 0.87300
# acc: 0.87830
# d = 4096
# acc: 0.72260
# acc: 0.77610
# acc: 0.78610
# acc: 0.79670
# acc: 0.80480
# acc: 0.80960
# acc: 0.81540
# acc: 0.81800
# acc: 0.82110
# acc: 0.82140

# d = 3072
# acc: 0.68250
# acc: 0.74680
# acc: 0.77790
# acc: 0.78910
# acc: 0.79730
# acc: 0.80230
# acc: 0.80830
# acc: 0.80790
# acc: 0.81220
# acc: 0.81380

# d = 2048
# acc: 0.66780
# acc: 0.72230
# acc: 0.74850
# acc: 0.76220
# acc: 0.77360
# acc: 0.78370
# acc: 0.78950
# acc: 0.79590
# acc: 0.80070
# acc: 0.80140

# d = 1024
# acc: 0.64890
# acc: 0.68100
# acc: 0.71570
# acc: 0.73890
# acc: 0.74800
# acc: 0.75690
# acc: 0.76260
# acc: 0.76620
# acc: 0.76720
# acc: 0.77080

# d = 512
# acc: 0.53550
# acc: 0.60120
# acc: 0.65100
# acc: 0.67860
# acc: 0.69580
# acc: 0.70770
# acc: 0.71970
# acc: 0.72760
# acc: 0.73310
# acc: 0.73640

# d = 256
# acc: 0.30830
# acc: 0.48570
# acc: 0.55060
# acc: 0.59290
# acc: 0.61540
# acc: 0.62600
# acc: 0.63280
# acc: 0.64080
# acc: 0.65070
# acc: 0.66000
# # d = 128
# acc: 0.11780
# acc: 0.20260
# acc: 0.31380
# acc: 0.40520
# acc: 0.45310
# acc: 0.48530
# acc: 0.50790
# acc: 0.52710
# acc: 0.54030
# acc: 0.55170





In [ ]:
import matplotlib.pyplot as plt


plt


In [ ]:
import matplotlib.pyplot as plt


epochs = list(range(1, 11))
# yes , i know the list like these are awful to look, and modify . ; I am aware of that fact :p
baseline = [0.83080, 0.85580, 0.85860, 0.86500, 0.86980, 0.87200, 0.87390, 0.87400, 0.87300, 0.87830]
d_4096   = [0.72260, 0.77610, 0.78610, 0.79670, 0.80480, 0.80960, 0.81540, 0.81800, 0.82110, 0.82140]
d_3072   = [0.68250, 0.74680, 0.77790, 0.78910, 0.79730, 0.80230, 0.80830, 0.80790, 0.81220, 0.81380]
d_2048   = [0.66780, 0.72230, 0.74850, 0.76220, 0.77360, 0.78370, 0.78950, 0.79590, 0.80070, 0.80140]
d_1024   = [0.64890, 0.68100, 0.71570, 0.73890, 0.74800, 0.75690, 0.76260, 0.76620, 0.76720, 0.77080]
d_512    = [0.53550, 0.60120, 0.65100, 0.67860, 0.69580, 0.70770, 0.71970, 0.72760, 0.73310, 0.73640]
d_256    = [0.30830, 0.48570, 0.55060, 0.59290, 0.61540, 0.62600, 0.63280, 0.64080, 0.65070, 0.66000]
d_128    = [0.11780, 0.20260, 0.31380, 0.40520, 0.45310, 0.48530, 0.50790, 0.52710, 0.54030, 0.55170]

final_baseline = baseline[-1]
threshold_value = 0.9 * final_baseline


plt.figure(figsize=(12, 7))
plt.grid(True, linestyle='--', alpha=0.5)


plt.plot(epochs, baseline, label='Baseline', marker='o', linewidth=3, color='black')
plt.plot(epochs, d_4096,   label='d = 4096', marker='v', linestyle='-')
plt.plot(epochs, d_3072,   label='d = 3072', marker='p', linestyle='-')
plt.plot(epochs, d_2048,   label='d = 2048', marker='h', linestyle='-')
plt.plot(epochs, d_1024,   label='d = 1024', marker='s', linestyle='--')
plt.plot(epochs, d_512,    label='d = 512',  marker='^', linestyle='--')
plt.plot(epochs, d_256,    label='d = 256',  marker='d', linestyle=':')
plt.plot(epochs, d_128,    label='d = 128',  marker='x', linestyle=':')


plt.axhline(y=threshold_value, color='red', linestyle='-.', linewidth=2, 
            label=f'90% Baseline Target ({threshold_value:.4f})')


plt.title('Model Accuracy vs. Dimension d with 90% Baseline Target', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.xticks(epochs)
plt.ylim(0.0, 1.0)


plt.legend(loc='upper left', bbox_to_anchor=(1, 1), fontsize=10)
plt.tight_layout()


plt.show()
